<a href="https://colab.research.google.com/github/devyadav11/ML_tryouts/blob/main/LLM_PPT_AutoGenerate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)



In [5]:
# ================================
# 🔹 Install required libraries
# ================================
!pip install duckduckgo-search python-pptx groq
!pip install ddgs

# ================================
# 🔹 Imports
# ================================
import os
import json
from groq import Groq
from duckduckgo_search import DDGS
from pptx import Presentation
from pptx.util import Pt
from google.colab import userdata

# ================================
# 🔹 Set Groq API Key
# ================================
# Get one free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# ================================
# 🔹 Topic input
# ================================
topic = input("Enter the topic for the PowerPoint slide deck: ")

# Perform web search
search_query = f"{topic} key facts and latest developments"
with DDGS() as ddgs:
    search_results = [r for r in ddgs.text(search_query, max_results=10)]

search_summary = "\n\n".join(
    [f"Title: {r['title']}\nSnippet: {r['body']}\nURL: {r['href']}" for r in search_results]
)

# ================================
# 🔹 LLM Prompt
# ================================
llm_prompt = f"""
Using your internal knowledge and the following web search results for up-to-date information:

{search_summary}

Generate a structured PowerPoint slide deck content on the topic: '{topic}'.

The deck should have exactly 7 slides:
- Slide 1: Title (main title and a subtitle)
- Slide 2: Overview (brief summary in bullet points)
- Slides 3-6: Key points / trends / arguments (each slide focuses on one main aspect with 3-5 bullet points)
- Slide 7: Conclusion / Takeaways (key takeaways in bullet points)

Output strictly in JSON format without any additional text:
{{
  "slides": [
    {{"title": "Main Title", "subtitle": "Subtitle" }},
    {{"title": "Slide Title", "bullets": ["bullet1", "bullet2", ...] }}
  ]
}}
"""

# ================================
# 🔹 Call Groq (free alternative)
# ================================
response = groq_client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[{"role": "user", "content": llm_prompt}],
    temperature=0.7,
    max_tokens=2000
)

import re, json

# Get raw content string
raw_output = response.choices[0].message.content

# Extract the JSON block safely
json_match = re.search(r"\{.*\}", raw_output, re.DOTALL)
if json_match:
    generated_content = json.loads(json_match.group(0))
else:
    raise ValueError("No valid JSON found in model response")


# ================================
# 🔹 Create PowerPoint
# ================================
prs = Presentation()

# Title slide
title_slide_layout = prs.slide_layouts[0]
title_slide = prs.slides.add_slide(title_slide_layout)
title = title_slide.shapes.title
title.text = generated_content["slides"][0]["title"]
subtitle = title_slide.placeholders[1]
subtitle.text = generated_content["slides"][0]["subtitle"]

# Other slides
for slide_data in generated_content["slides"][1:]:
    slide_layout = prs.slide_layouts[1]  # Title + Content
    slide = prs.slides.add_slide(slide_layout)
    title = slide.shapes.title
    title.text = slide_data["title"]

    tf = slide.placeholders[1].text_frame
    for bullet in slide_data["bullets"]:
        p = tf.add_paragraph()
        p.text = bullet
        p.space_before = Pt(12)
        p.space_after = Pt(12)

# Save file
output_filename = f"{topic.replace(' ', '_')}_Slide_Deck.pptx"
prs.save(output_filename)
print(f"✅ PowerPoint slide deck saved as: {output_filename}")


Enter the topic for the PowerPoint slide deck: CLIMATE


/tmp/ipython-input-3331232183.py:32: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


✅ PowerPoint slide deck saved as: CLIMATE_Slide_Deck.pptx


In [1]:
!pip install --upgrade jupyter_client
